# 10. Feature Selection

This notebook performs feature selection on the encoded and pre-filtered dataset to identify the most relevant features for predicting student dropout.

## Objectives
- Load encoded and pre-filtered data from previous steps
- Explore dataset characteristics and structure
- Perform feature selection techniques to reduce dimensionality
- Evaluate feature importance and select optimal subset
- Prepare final feature set for model training

## Table of Contents
1. [Setup & Data Loading](#setup--data-loading)
2. [Data Exploration](#data-exploration)
3. [Feature Selection Methods](#feature-selection-methods)
   - 3.1 [Variance Threshold Filtering](#variance-threshold-filtering)
4. [Feature Evaluation](#feature-evaluation)
5. [Final Feature Set](#final-feature-set)

---

## 1. Setup & Data Loading {#setup--data-loading}

Import required libraries and load the encoded dataset from previous preprocessing steps.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold

In [ ]:
# Define paths
current_user = os.getlogin()
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")
OUT_DIR = PROCESSED_DIR

# Load data
data = pd.read_csv(PROCESSED_DIR / "encoded_and_split_data.csv", sep=None, engine="python", encoding="utf-8-sig")
print(f"Data shape: {data.shape}")
data.head()

## 2. Data Exploration {#data-exploration}

Examine the structure and characteristics of the loaded dataset before applying feature selection.

In [ ]:
# Basic data exploration
print(f"Dataset shape: {data.shape}")
print(f"Columns: {len(data.columns)}")
print(f"Memory usage: {data.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Check data types
print(f"\nData types distribution:")
print(data.dtypes.value_counts())

# Check for missing values
missing_summary = data.isnull().sum()
missing_cols = missing_summary[missing_summary > 0]
if len(missing_cols) > 0:
    print(f"\nColumns with missing values: {len(missing_cols)}")
    print(missing_cols.head(10))
else:
    print("\nNo missing values found")

# Check target distribution if available
if 'is_dropout' in data.columns:
    print(f"\nTarget distribution:")
    print(data['is_dropout'].value_counts())

## 3. Feature Selection Methods {#feature-selection-methods}

Apply feature selection techniques to identify the most important features for dropout prediction:
- variance threshold filtering
- feature correlation
- permutation importance

Feature selection is based on the training dataset only.

In [ ]:
# Columns to drop based on feature selection
columns_to_drop = [
    # Add columns to drop here after feature selection
    'attended_online_opleidingspresentatie',                        # Feature does not have a value in training set
    'sdo2_orientation_First_Event_Date_missing_flag',               # High correlation with has_orientation
    'sdo2_orientation_Last_Event_Date_missing_flag',                # High correlation with has_orientation
    'gap_skc_to_start_weeks_missing_flag',                          # High correlation with has_skc
    'sdo2_skc_ADVIES_DEF_Niet_deelgenomen',                         # Is identical to has_skc feature
    'sdo2_orientation_Number_of_Events_Attended_missing_flag',      # Is identical to has_orientation feature
    'sdo2_orientation_Number_of_Event_Types_missing_flag',          # Is identical to attended_orientation_type_absent feature
    'attended_orientation_type_absent',                             # Is identical to sdo2_orientation_Number_of_Event_Types feature
    'sdo1_previous_Final_Exam_Date_missing_flag',                   # Is identical to has_previous feature
    'sdo1_previous_Previous_Education_Type_Unknown',                # Is almost identical to has_previous feature
    'sdo1_postal_distance_distance_to_3584_CS_missing_flag',        # High correlation with sdo5_degree_POSTAL_COUNTRY_NL feature
    'has_orientation',                                              # Is redundant since sdo2_orientation_Number_of_Events_Attended already encodes this information
    'time_first_orient_to_start_weeks_missing_flag',                # High correlation with sdo2_orientation_Number_of_Events_Attended
    'time_last_orient_to_start_weeks_missing_flag',                 # High correlation with sdo2_orientation_Number_of_Events_Attended
    'sdo6_results_Percentage_Credits_A',                            # Remove derived feature, since it might introduce multicollinearity with sdo6_results_Credits_Block_A and/or sdo6_results_Potential_Credits_A
    'sdo6_results_Percentage_Credits_A_missing_flag',               # Feature derived is removed above
    'SINH_ID'                                                       # Only needed for testing-purposes
]
data = data.drop(columns=columns_to_drop, errors='ignore')

In [ ]:
# Select training data
data_train = data[data['set'] == 'train']
data_train

Define which columns to ignore (these are not considered features):

In [ ]:
columns_to_ignore = ['set']
data_train = data_train.drop(columns=columns_to_ignore, errors='ignore')

### 3.1 Variance Threshold Filtering

Removes features with little to no variation.

In [ ]:
# Investigate features with little to no variation
# For each numeric column, calculate variance

# Identify numeric columns (exclude string/object columns)
numeric_cols = data_train.select_dtypes(include=[np.number]).columns.tolist()

# Calculate variance for each numeric column
variance_analysis = pd.DataFrame({
    'column': numeric_cols,
    'variance': [data_train[col].var() for col in numeric_cols],
    'std_dev': [data_train[col].std() for col in numeric_cols],
    'unique_values': [data_train[col].nunique() for col in numeric_cols],
    'data_type': [str(data_train[col].dtype) for col in numeric_cols]
}).sort_values('variance')

print(f"📊 Variance Analysis for {len(numeric_cols)} numeric features:\n")

# Show features with lowest variance (potential candidates for removal)
print("🔻 Features with LOWEST variance (top 20):")
print(variance_analysis.head(20)[['column', 'variance', 'unique_values']].to_string(index=False))

print(f"\n🔺 Features with HIGHEST variance (top 10):")
print(variance_analysis.tail(10)[['column', 'variance', 'unique_values']].to_string(index=False))

# Identify zero-variance features
zero_variance = variance_analysis[variance_analysis['variance'] == 0.0]
print(f"\n⚠️  Features with ZERO variance: {len(zero_variance)}")
if len(zero_variance) > 0:
    print(zero_variance[['column', 'unique_values']].to_string(index=False))

# Identify very low variance features (< 0.01)
low_variance = variance_analysis[
    (variance_analysis['variance'] < 0.01) & (variance_analysis['variance'] > 0)
]
print(f"\n⚠️  Features with very LOW variance (< 0.01): {len(low_variance)}")
if len(low_variance) > 0:
    print(low_variance[['column', 'variance', 'unique_values']].to_string(index=False))

### 3.2 Feature correlation
Examines pairs of highly correlated feature pairs. 

In [ ]:
# Create hybrid correlation matrix: Pearson for binary-binary, Spearman for others
def create_hybrid_correlation_matrix(data, numeric_cols):
    """
    Create correlation matrix using:
    - Pearson correlation for binary-binary feature pairs
    - Spearman correlation for all other combinations
    """
    # Initialize with Spearman correlation as base
    correlation_matrix = data[numeric_cols].corr(method='spearman')
    
    # Identify binary columns (only values 0 and 1)
    binary_cols = []
    for col in numeric_cols:
        unique_vals = set(data[col].dropna().unique())
        if unique_vals.issubset({0, 1}):
            binary_cols.append(col)
    
    print(f"📊 Identified {len(binary_cols)} binary columns for Pearson correlation")
    
    # Calculate Pearson correlation for binary-binary pairs
    if len(binary_cols) > 1:
        pearson_matrix = data[binary_cols].corr(method='pearson')
        
        # Replace binary-binary correlations with Pearson values
        for i, col1 in enumerate(binary_cols):
            for j, col2 in enumerate(binary_cols):
                if i != j:  # Don't replace diagonal (self-correlation)
                    correlation_matrix.loc[col1, col2] = pearson_matrix.loc[col1, col2]
    
    return correlation_matrix

# Apply hybrid correlation method
correlation_matrix = create_hybrid_correlation_matrix(data_train, numeric_cols)

print("🔄 Hybrid correlation matrix created:")
print("   - Pearson correlation for binary-binary feature pairs")
print("   - Spearman correlation for all other combinations")

# Visualize correlation matrix (optional)
import seaborn as sns
import matplotlib.pyplot as plt 
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, square=True)
plt.title("Feature Correlation Matrix")
plt.show()

# Print highly correlated feature pairs
threshold = 0.9
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if correlation_matrix.iloc[i, j] > threshold or correlation_matrix.iloc[i, j] < -threshold:
            high_corr_pairs.append((correlation_matrix.columns[i], correlation_matrix.columns[j], correlation_matrix.iloc[i, j]))

print(f"\n⚠️  Highly correlated feature pairs (correlation > {threshold}): {len(high_corr_pairs)}")
for pair in high_corr_pairs:
    print(f"{pair[0]} and {pair[1]}: correlation = {pair[2]:.2f}")

`Has_nf` and `has_skc` are expected to strongly correlate (negatively): all students enrolling in a numerus fixus degree (`has_nf = 1`) do not participate in the skc (`has_skc = 0`). The correlation is not perfect since there are many students that for other reasons have no skc data available, keeping both features is therefore advised. 

#### 3.2.1 Correlation with outcome (predictor) feature

In [ ]:
# Look at corrections with target variable sdo5_degree_drop_out
target_variable = 'sdo5_degree_drop_out'

target_correlations = correlation_matrix[target_variable].sort_values(ascending=False)
print(f"\n🔍 Correlation of features with target variable '{target_variable}' (desc):")
print(target_correlations.to_string())

### 3.3 Permutation Importance (Random Forest)
Capture nonlinear and interaction-based importance

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score
import warnings

# Perform permutation importance using random forest
warnings.filterwarnings('ignore')

# Prepare target and features
X_train = data_train.drop('sdo5_degree_drop_out', axis=1)
y_train = data_train['sdo5_degree_drop_out']

# Handle any remaining object columns by dropping them (dates)
object_cols = X_train.select_dtypes(include=['object']).columns
X_train = X_train.drop(columns=object_cols)

print(f"Training data shape: {X_train.shape}")
print(f"Target distribution:\n{y_train.value_counts()}")

# Train Random Forest model
print("\n🌲 Training Random Forest model...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
rf_model.fit(X_train, y_train)

# Calculate permutation importance
print("\n🔄 Calculating permutation importance...")
perm_importance = permutation_importance(
    rf_model, X_train, y_train,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

# Create dataframe with results
perm_importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=False)

# Display top important features
print("\n📊 Top 20 most important features (Permutation Importance):")
print(perm_importance_df.head(20).to_string(index=False))

print(f"\n📊 Bottom 20 least important features:")
print(perm_importance_df.tail(20).to_string(index=False))

# Visualize top features
plt.figure(figsize=(10, 8))
top_n = 20
top_features = perm_importance_df.head(top_n)
plt.barh(range(len(top_features)), top_features['importance_mean'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Permutation Importance')
plt.title(f'Top {top_n} Features by Permutation Importance (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 4. Feature Evaluation {#feature-evaluation}

Global / model-agnostic feature selection

This script computes cross-validated permutation feature
importance and uses it as a global feature selection
technique that is largely model-agnostic. The goal is not to
optimize features for a specific model via iterative subset
selection (e.g., RFE), but to identify features that show
consistent predictive value on unseen data and can therefore
be shared across different model families.

Why permutation importance (ROC-AUC based):

Permutation importance measures the drop in ROC-AUC when a
feature column is randomly shuffled, breaking its
relationship with the target while keeping all other
features unchanged. A larger drop indicates that the model
relied on that feature to separate the classes.

Using ROC-AUC provides a threshold-independent and
class-balanced measure of feature contribution, making it
suitable for imbalanced classification and global feature
assessment.

Evaluating importance on held-out validation folds helps
identify features that contribute to generalization, not
just training fit.

Role of stability (robustness across folds):

Stability measures how consistently a feature contributes
positively across cross-validation folds, complementing
average permutation importance with a robustness criterion.

With 5-fold cross-validation:
stability = 1.0 -> helpful in all 5 folds
stability = 0.8 -> helpful in 4 out of 5 folds
stability = 0.6 -> helpful in 3 out of 5 folds
stability = 0.2 -> helpful in only 1 fold
stability = 0.0 -> never helpful

This interpretation makes stability an intuitive measure
of reproducibility and guards against fold-specific noise,
rare-category effects, and spurious correlations.

Feature elimination:

Features are marked for removal only when they show no
average contribution (non-positive mean importance) and
low stability (at most one fold) across folds, ensuring that only consistently
non-informative features are dropped while avoiding
over-pruning.

In [ ]:
# ============================================================
# Global, model-agnostic feature selection using
# cross-validated permutation importance + stability
# ============================================================


# ------------------------------------------------------------
# Prepare target and features (TRAINING DATA ONLY)
# ------------------------------------------------------------
X = data_train.drop("sdo5_degree_drop_out", axis=1)
y = data_train["sdo5_degree_drop_out"]

# Drop remaining object columns (e.g., dates stored as strings)
object_cols = X.select_dtypes(include=["object"]).columns
X = X.drop(columns=object_cols)

print(f"Training data shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

# ------------------------------------------------------------
# Random Forest used only as an importance estimator
# ------------------------------------------------------------
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

# ------------------------------------------------------------
# Cross-validated permutation importance (evaluated on CV folds)
# ------------------------------------------------------------
print("\n🔁 Calculating CV-based permutation importance...")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_importances = []  # list of arrays (n_features,)
for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Fit on fold-train
    rf_model.fit(X_tr, y_tr)

    # Permutation importance evaluated on fold-validation (unseen)
    pi = permutation_importance(
        rf_model,
        X_val,
        y_val,
        n_repeats=10,
        random_state=42,
        n_jobs=-1,
        scoring="roc_auc"  # robust choice for imbalanced classification
    )

    fold_importances.append(pi.importances_mean)
    print(f"   ✅ Fold {fold} done")

fold_importances = np.vstack(fold_importances)  # shape: (n_folds, n_features)

# ------------------------------------------------------------
# Aggregate importance across folds
# ------------------------------------------------------------
importance_mean_cv = fold_importances.mean(axis=0)
importance_std_cv = fold_importances.std(axis=0)

# Stability = proportion of folds where importance > 0
stability = (fold_importances > 0).mean(axis=0)

perm_importance_df = (
    pd.DataFrame({
        "feature": X.columns,
        "importance_mean": importance_mean_cv,
        "importance_std": importance_std_cv,
        "stability": stability
    })
    .sort_values("importance_mean", ascending=False)
)

# ------------------------------------------------------------
# Inspect results
# ------------------------------------------------------------
print("\n📊 Top 20 most important features (CV Permutation Importance):")
print(perm_importance_df.head(20).to_string(index=False))

print("\n📊 Bottom 20 least important features:")
print(perm_importance_df.tail(20).to_string(index=False))

# ------------------------------------------------------------
# Visualization
# ------------------------------------------------------------
plt.figure(figsize=(10, 8))
top_n = 20
top_features = perm_importance_df.head(top_n).iloc[::-1]
plt.barh(top_features["feature"], top_features["importance_mean"])
plt.xlabel("Mean CV Permutation Importance (ROC-AUC")
plt.title(f"Top {top_n} Features by CV Permutation Importance (Random Forest)")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Conservative feature dropping
# ------------------------------------------------------------
# Drop features that:
#  - have no average contribution (importance_mean <= 0)
#  - are helpful in at most 1 out of 5 folds (stability <= 0.2)
drop_features = perm_importance_df[
    (perm_importance_df["importance_mean"] <= 0) &
    (perm_importance_df["stability"] <= 0.2)
]["feature"].tolist()

print(f"\n Suggested drop candidates (conservative): {len(drop_features)}")
if drop_features:
    for f in drop_features:
        print(f"  - {f}")
else:
    print("  None")

In [ ]:
# ============================================================
# Apply global feature selection: drop selected features
# (using drop_features computed from CV permutation importance)
# ============================================================

print(f"🔻 Dropping {len(drop_features)} features from data:")
for f in drop_features:
    print(f"  - {f}")

# Drop features from data only
data = data.drop(columns=drop_features, errors="ignore")

print("\n✅ Feature drop complete")
print(f"   New data shape: {data.shape}")

## 5. Final Feature Set {#final-feature-set}

Select the optimal feature subset and prepare the final dataset for model training.

In [ ]:
# Save dataset
output_path = OUT_DIR / "feature_selection.csv"
data.to_csv(output_path, index=False)

print("✅ Feature selection complete!")
print(f"   📂 Saved: {output_path}")
print(f"   📊 Shape: {data.shape}")
print(f"   🏷️ Columns: {len(data.columns)}")

# Show data type summary
print(f"\n📋 Data type summary:")
dtype_counts = data.dtypes.value_counts()
for dtype, count in dtype_counts.items():
    print(f"   {dtype}: {count} columns")